# Drug Prescription Guide
## Extracting rules from a Decision Tree

**Goal:** train a Decision Tree on the drug dataset and convert its internal
paths into a human-readable guide for doctors — with original feature labels,
no encoded numbers.

We build everything in 6 steps, understanding each piece before moving on.

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## 0. Raw data

200 patients with the same illness, each responding to one of 5 drugs.

| Feature | Type | Values |
|---|---|---|
| Age | Numerical | integer |
| Sex | Categorical | F, M |
| BP | Categorical | HIGH, LOW, NORMAL |
| Cholesterol | Categorical | HIGH, NORMAL |
| Na_to_K | Numerical | float |

In [2]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/drug200.csv'
df = pd.read_csv(url)
df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY


## 1. Preprocessing — saving the decoder

The model needs numbers. We use `LabelEncoder` on categorical features, but
this time we also save the reverse mapping in `cat_decoders`: a list of labels
ordered by the numeric index the encoder assigned.

This is what will let us translate `BP <= 0.5` back to `BP = HIGH` later.

> **Why `.5` thresholds?** sklearn uses half-integer thresholds (0.5, 1.5, ...)
> for integer-encoded features so that all integer values fall cleanly on one
> side of the cut.

In [ ]:
CAT_FEATURES = ['Sex', 'BP', 'Cholesterol']

df_model = df.copy()
cat_decoders = {}  # {feature: [label_for_index_0, label_for_index_1, ...]}

for col in CAT_FEATURES:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    cat_decoders[col] = list(le.classes_)  # LabelEncoder sorts alphabetically

print('cat_decoders:')
for feat, labels in cat_decoders.items():
    mapping = {i: v for i, v in enumerate(labels)}
    print(f'  {feat}: {mapping}')

## 2. Train the Decision Tree

In [ ]:
X = df_model.drop('Drug', axis=1)
y = df_model['Drug']
feature_names = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=32)

clf = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=32)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f'Accuracy: {acc * 100:.2f}%')

## 3. What is inside `clf.tree_`?

sklearn stores the entire tree structure in **arrays indexed by node ID**.
Each position `i` in these arrays describes node `i`:

| Array | What it holds |
|---|---|
| `tree_.feature[i]` | Index of the feature used at node `i` (`-2` = leaf) |
| `tree_.threshold[i]` | Split threshold at node `i` |
| `tree_.children_left[i]` | Node ID of the left child (samples where feature <= threshold) |
| `tree_.children_right[i]` | Node ID of the right child (samples where feature > threshold) |
| `tree_.value[i]` | Sample count per class at node `i` |

Let's inspect:

In [ ]:
t = clf.tree_
classes_header = '  '.join(f'{c:>5}' for c in clf.classes_)

print(f"{'Node':>4}  {'Feature':>14}  {'Threshold':>10}  {'<=':>5}  {'>':>5}  {classes_header}")
print('-' * 75)

for i in range(t.node_count):
    feat_name = feature_names[t.feature[i]] if t.feature[i] != -2 else 'LEAF'
    thresh    = f'{t.threshold[i]:.3f}' if t.threshold[i] != -2.0 else '--'
    left      = str(t.children_left[i])  if t.children_left[i]  != -1 else '--'
    right     = str(t.children_right[i]) if t.children_right[i] != -1 else '--'
    counts    = '  '.join(f'{int(v):>5}' for v in t.value[i][0])
    print(f'{i:>4}  {feat_name:>14}  {thresh:>10}  {left:>5}  {right:>5}  {counts}')

## 4. `decode_condition` — translating thresholds back to original labels

The problem without decoding:
```
BP <= 0.5   <- what does this mean to a doctor?
```

With `cat_decoders` we know BP was encoded as `{HIGH:0, LOW:1, NORMAL:2}`.
So `BP <= 0.5` means "BP index <= 0", which is only **HIGH**.

**General logic:**
- `int(threshold)` = maximum index that falls on the **left** side (floor of the .5)
- Left side (<=): labels from index 0 up to `int(threshold)` inclusive
- Right side (>): remaining labels

In [ ]:
def decode_condition(feature, threshold, direction, cat_decoders, round_digits=3):
    # Numerical feature: returns 'Na_to_K <= 14.627' or 'Age > 50.500'
    if feature not in cat_decoders:
        op = '<=' if direction == 'left' else '>'
        return f'{feature} {op} {threshold:.{round_digits}f}'

    labels = cat_decoders[feature]
    max_left_idx = int(threshold)  # floor: max index on the left side

    if direction == 'left':
        matched = labels[:max_left_idx + 1]
    else:
        matched = labels[max_left_idx + 1:]

    if len(matched) == 1:
        return f'{feature} = {matched[0]}'
    return f'{feature} in [{", ".join(matched)}]'

In [ ]:
# Testing every possible case with BP and Cholesterol
print(decode_condition('BP',          0.5,    'left',  cat_decoders))  # BP = HIGH
print(decode_condition('BP',          0.5,    'right', cat_decoders))  # BP in [LOW, NORMAL]
print(decode_condition('BP',          1.5,    'right', cat_decoders))  # BP = NORMAL
print(decode_condition('Cholesterol', 0.5,    'left',  cat_decoders))  # Cholesterol = HIGH
print(decode_condition('Na_to_K',     14.627, 'left',  cat_decoders))  # Na_to_K <= 14.627
print(decode_condition('Age',         50.5,   'right', cat_decoders))  # Age > 50.500

## 5. `traverse_tree` — walking the tree recursively

Core algorithm. The function starts at node 0 (root) with an empty condition
list and walks down every path to the leaves:

```
root (node 0)
 +-- Na_to_K <= 14.627  (node 1)
 |    +-- BP = HIGH  (node 2)
 |    |    +-- Age <= 50.500  ->  drugA
 |    |    +-- Age > 50.500   ->  drugB
 |    +-- BP in [LOW, NORMAL]  (node 5)
 |         +-- Cholesterol = HIGH  (node 6)
 |         |    +-- ...  ->  drugC
 |         |    +-- ...  ->  drugX
 |         +-- Cholesterol = NORMAL  ->  drugX
 +-- Na_to_K > 14.627  ->  drugY
```

At each internal node: accumulate the decoded condition and recurse into
both children. At a leaf: record (drug, conditions).

In [ ]:
def traverse_tree(node_id, conditions, clf, feature_names, cat_decoders, paths):
    t     = clf.tree_
    left  = t.children_left[node_id]
    right = t.children_right[node_id]

    # Leaf: record the predicted drug for this path
    if left == _tree.TREE_LEAF:
        class_idx  = t.value[node_id][0].argmax()
        class_name = clf.classes_[class_idx]
        paths.append((class_name, list(conditions)))
        return

    feat   = feature_names[t.feature[node_id]]
    thresh = t.threshold[node_id]

    left_cond  = decode_condition(feat, thresh, 'left',  cat_decoders)
    right_cond = decode_condition(feat, thresh, 'right', cat_decoders)

    traverse_tree(left,  conditions + [left_cond],  clf, feature_names, cat_decoders, paths)
    traverse_tree(right, conditions + [right_cond], clf, feature_names, cat_decoders, paths)

In [ ]:
# Raw output — all paths before formatting
paths = []
traverse_tree(0, [], clf, feature_names, cat_decoders, paths)

for drug, conds in paths:
    print(f'{drug}: {conds}')

## 6. `generate_drug_guide` — the final guide

Group the collected paths by drug and format the output.

> **Note:** `drugX` appears under more than one rule because there are two
> distinct paths in the tree that lead to it. That is expected.

In [ ]:
def generate_drug_guide(clf, feature_names, cat_decoders, accuracy=None):
    paths = []
    traverse_tree(0, [], clf, feature_names, cat_decoders, paths)

    by_drug = defaultdict(list)
    for drug, conds in paths:
        by_drug[drug].append(conds)

    W   = 58
    EQ  = '=' * W
    SEP = '-' * W

    print(EQ)
    print('  DRUG PRESCRIPTION GUIDE -- Decision Tree')
    if accuracy is not None:
        print(f'  Model accuracy: {accuracy:.2%}')
    print(EQ)

    for drug in sorted(by_drug):
        rules = by_drug[drug]
        print()
        print(SEP)
        print(f'  {drug.upper()}')
        print(SEP)

        if len(rules) == 1:
            for cond in rules[0]:
                print(f'  * {cond}')
        else:
            for i, rule in enumerate(rules, 1):
                print(f'  Rule {i}:')
                for cond in rule:
                    print(f'    * {cond}')

    print()
    print(EQ)
    return by_drug

In [ ]:
acc = accuracy_score(y_test, clf.predict(X_test))
guide = generate_drug_guide(clf, feature_names, cat_decoders, accuracy=acc)